# 2024年版ものづくり白書 RAG チャットボット (クイック実行・実験向け v2版)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

本ノートブックは、**「2024年版ものづくり白書（全340ページ・31MB）」**を対象に、チャンク分割サイズの調整や検索方式の比較を効率よく行うための**集約・クイック実行版**です。

### 🎯 本ノートブックの構成 (全5ステップ)
1. **環境準備 (一括実行)**: 必要なライブラリのインストール、APIキー設定、白書PDFの自動取得
2. **チャンク分割パラメータ調整 & 集計**: Colabスライダーで `CHUNK_SIZE` や `CHUNK_OVERLAP` を動的変更
3. **検索エンジンの統合構築 (一括実行)**: BM25 ＋ ChromaDBローカルベクトル ＋ ハイブリッドRRF
4. **Gradio Web チャットボットUIの起動**: 3方式リアルタイム切替＆出典付き対話画面
5. **【コード単体検証】Geminiによる回答生成テスト**: プログラマティックな詳細デバッグ・根拠コンテキスト確認

## 1. 環境準備（ライブラリ + APIキー + 白書PDFの配置を一括実行）
以下のセルを1回実行するだけで、必要なライブラリのインストール、Gemini APIキーの設定、白書PDFのダウンロードが自動完了します。

In [ ]:
# 1. 必要なライブラリのインストール
print("[1/3] 📦 ライブラリをインストール中...")
!pip install -q --no-warn-conflicts \
    langchain \
    langchain-community \
    langchain-google-genai \
    langchain-chroma \
    langchain-huggingface \
    sentence-transformers \
    chromadb \
    pypdf \
    gradio \
    rank-bm25

# 2. Gemini API キーの設定
print("[2/3] 🔑 GEMINI_API_KEY を設定中...")
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get('GEMINI_API_KEY')
    print("  ✅ Colabシークレットから GEMINI_API_KEY を設定しました。")
except Exception:
    from getpass import getpass
    if "GEMINI_API_KEY" not in os.environ or not os.environ["GEMINI_API_KEY"]:
        os.environ["GEMINI_API_KEY"] = getpass("  Gemini API Key を入力してください: ")
    else:
        print("  ✅ 既存の環境変数から GEMINI_API_KEY を確認しました。")

# 3. 白書PDF（31MB）の配置
print("[3/3] 📄 白書PDFの準備中...")
os.makedirs("data", exist_ok=True)
pdf_target = "data/document.pdf"
if not os.path.exists(pdf_target):
    print(f"  [*] 経産省公式サイトからダウンロード中 ({pdf_target})...")
    !wget -q https://www.meti.go.jp/report/whitepaper/mono/2024/honbun/pdf/all.pdf -O data/document.pdf

if os.path.exists(pdf_target):
    size_mb = os.path.getsize(pdf_target) / (1024 * 1024)
    print(f"  ✅ 白書PDFを確認しました: {pdf_target} ({size_mb:.2f} MB)")
else:
    print("  ⚠️ 左メニューの📁ファイルから31MBの白書PDFをドラッグ＆ドロップして 'data/document.pdf' に配置してください。")

print("\n🎉 【ステップ1】環境準備がすべて完了しました！ 次のセルへお進みください。")

## 2. 階層メタデータ（章・節・ページ）付きチャンク分割 & pandas集計
白書の「1節丸ごとだと長すぎる」問題を解決するため、**章・節見出しをメタデータとして保持しながら指定した文字数単位に分割**します。

※ 右側のフォーム（スライダー）で `CHUNK_SIZE` や `CHUNK_OVERLAP` を自由に変更できます。
**数値を変更した場合は、このセルを実行後、必ず次（ステップ3）の検索エンジン構築セルも再実行してください。**

In [ ]:
#@title ⚙️ チャンク分割パラメータの設定 { run: "auto" }
CHUNK_SIZE = 700      #@param {type:"slider", min:300, max:1500, step:50}
CHUNK_OVERLAP = 150   #@param {type:"slider", min:50, max:400, step:10}

import os
import re
import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def extract_headings(text, current):
    meta = current.copy()
    p_match = re.search(r'(第[一二三四五六七八九十\d]+部\s+[^\n]{2,30})', text)
    if p_match: meta['part'] = p_match.group(1).strip()
    c_match = re.search(r'(第[一二三四五六七八九十\d]+章\s+[^\n]{2,35})', text)
    if c_match: meta['chapter'] = c_match.group(1).strip()
    s_match = re.search(r'(第[一二四五六七八九十\d]+節\s+[^\n]{2,35})', text)
    if s_match: meta['section'] = s_match.group(1).strip()
    return meta

pdf_target = "data/document.pdf"
if not os.path.exists(pdf_target):
    print("⚠️ 'data/document.pdf' が見つかりません。ステップ1を実行してください。")
else:
    print(f"[*] チャンク分割を実行中... (CHUNK_SIZE={CHUNK_SIZE}, CHUNK_OVERLAP={CHUNK_OVERLAP})")
    loader = PyPDFLoader(pdf_target)
    pages = loader.load()
    print(f"📖 読込ページ数: {len(pages)} ページ")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", "。", "、", " ", ""]
    )

    chunks = []
    records = []
    curr_meta = {'part': '第1部', 'chapter': '第1章', 'section': '第1節'}
    chunk_id = 0

    for p_doc in pages:
        p_num = p_doc.metadata.get('page', 0) + 1
        curr_meta = extract_headings(p_doc.page_content, curr_meta)
        p_chunks = splitter.split_text(p_doc.page_content)

        for txt in p_chunks:
            if len(txt.strip()) < 30: continue
            chunk_id += 1
            meta = {
                'chunk_id': chunk_id,
                'page': p_num,
                'part': curr_meta['part'],
                'chapter': curr_meta['chapter'],
                'section': curr_meta['section']
            }
            chunks.append(Document(page_content=txt.strip(), metadata=meta))
            records.append({
                'chunk_id': chunk_id,
                'page': p_num,
                'section': curr_meta['section'],
                'char_count': len(txt.strip()),
                'preview': txt.strip()[:60].replace('\n', ' ') + '...'
            })

    df_chunks = pd.DataFrame(records)
    print(f"✂️ 分割チャンク数: {len(chunks)} 件 (CHUNK_SIZE={CHUNK_SIZE})")
    print("\n📊 --- チャンクの文字数要約統計 (pandas describe) ---")
    display(df_chunks['char_count'].describe())
    print("\n📋 --- 検出された節ごとのチャンク数 (Top 5) ---")
    display(df_chunks['section'].value_counts().head(5))
    print("\n💡 【ご案内】チャンクサイズを変更した場合は、次のセル（検索エンジンの構築）を必ず再実行してください！")

## 3. 検索エンジンの構築（BM25 + ChromaDBローカル + RRFハイブリッドを一括構築）
以下の3つの検索エンジンを一度に構築します：
1. **キーワード検索 (BM25)**: 専門用語・固有名詞の完全一致
2. **ベクトル検索 (ChromaDB + sentence-transformers)**: 384次元のオープンソース多言語モデル（完全ローカル処理・API制限ゼロ）
3. **ハイブリッド検索 (RRF: Reciprocal Rank Fusion)**: キーワードとベクトルの順位を統合した高精度検索

> ⚠️ **ステップ2でチャンクサイズを変更した場合は、このセルを再実行してインデックスを更新してください。**

In [ ]:
import os
import shutil
import torch
from rank_bm25 import BM25Okapi
from langchain_chroma import Chroma
try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

os.environ["ANONYMIZED_TELEMETRY"] = "False"

if 'chunks' not in locals() or len(chunks) == 0:
    print("⚠️ チャンクデータがありません。先にステップ2を実行してください。")
else:
    # ---------------------------------------------------------
    # 1. キーワード検索 (BM25) のインデックス作成
    # ---------------------------------------------------------
    print("[1/3] 🔤 BM25キーワード検索インデックスを作成中...")
    def tokenize_jp(text):
        clean = text.lower()
        words = re.findall(r'[a-zA-Z0-9]+|[一-龥]{2,}|[ァ-ンヴー]{2,}', clean)
        bigrams = [clean[i:i+2] for i in range(len(clean)-1) if not re.match(r'\s', clean[i:i+2])]
        return list(set(words + bigrams)) or [clean]

    corpus_tokens = [tokenize_jp(c.page_content) for c in chunks]
    bm25 = BM25Okapi(corpus_tokens)
    print("  ✅ BM25キーワード検索エンジンの準備完了")

    # ---------------------------------------------------------
    # 2. ベクトル検索 (ChromaDB + sentence-transformers) の構築
    # ---------------------------------------------------------
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\n[2/3] 🧠 ローカル多言語埋め込みモデルを準備中 (Device: {device})...")
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": True}
    )

    # チャンクサイズ変更時に旧インデックスと混ざらないよう再生成
    persist_dir = "./chroma_db_v2"
    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir, ignore_errors=True)

    vector_store = Chroma(
        collection_name="whitepaper_vectors_v2",
        embedding_function=embeddings,
        persist_directory=persist_dir
    )

    total_chunks = len(chunks)
    BATCH_SIZE = 100
    print(f"  [*] ChromaDB に全 {total_chunks} 件をインデックス中 (完全ローカル処理 / API制限ゼロ)...")
    for i in range(0, total_chunks, BATCH_SIZE):
        batch = chunks[i : i + BATCH_SIZE]
        vector_store.add_documents(batch)
        done_count = min(i + len(batch), total_chunks)
        print(f"      進捗: {done_count}/{total_chunks} 件完了")
    print("  ✅ ベクトル検索エンジンの準備完了")

    # ---------------------------------------------------------
    # 3. ハイブリッド検索 (RRF) 関数の定義
    # ---------------------------------------------------------
    print("\n[3/3] 🏆 ハイブリッド検索 (RRF) 関数を定義中...")
    def hybrid_search(query, top_k=4, rrf_k=60, w_kw=0.5, w_vec=0.5):
        # キーワード検索
        kw_scores = bm25.get_scores(tokenize_jp(query))
        kw_top = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:top_k*2]

        # ベクトル検索
        vec_res = vector_store.similarity_search_with_relevance_scores(query, k=top_k*2)

        fused = {}
        for rank, idx in enumerate(kw_top, 1):
            if kw_scores[idx] <= 0: continue
            c_id = chunks[idx].metadata['chunk_id']
            fused[c_id] = {
                'doc': chunks[idx],
                'score': w_kw * (1.0 / (rrf_k + rank)),
                'kw_rank': rank,
                'vec_rank': None
            }

        for rank, (doc, sc) in enumerate(vec_res, 1):
            c_id = doc.metadata['chunk_id']
            score_val = w_vec * (1.0 / (rrf_k + rank))
            if c_id in fused:
                fused[c_id]['score'] += score_val
                fused[c_id]['vec_rank'] = rank
            else:
                fused[c_id] = {'doc': doc, 'score': score_val, 'kw_rank': None, 'vec_rank': rank}

        return sorted(fused.values(), key=lambda x: x['score'], reverse=True)[:top_k]

    print("  ✅ ハイブリッド検索エンジンの準備完了")
    print("\n🎉 3つの検索エンジンがすべて準備完了しました！")

## 4. Gradio Web チャットボットUI（検索方式切替 ＆ 出典付き回答）の起動
Colabセル内で対話できるWebチャット画面を起動します。
* **検索方式のリアルタイム切替**: 「ハイブリッド検索 (RRF)」「キーワード検索 (BM25)」「ベクトル検索 (ChromaDB)」を切り替えて回答を比較可能
* **最新モデル Gemini 3.8 Flash**: 白書の資料のみを根拠に、章・節・ページ番号（`[第X章 第X節 p.Y]`）を明記して回答
* **Gradio 5 最新仕様対応**: チャットメッセージ履歴を辞書形式で安全に取り扱い

In [ ]:
import gradio as gr
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Gemini 3.8 Flash モデルの準備
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0.2)

# 2. 出典明記プロンプトテンプレート
prompt_tmpl = """あなたは「ものづくり白書」の専門アシスタントです。
以下の【資料】のみを根拠にして質問に回答してください。
資料に記載のない内容は推測せず「資料に記載がありません」と答え、回答文中に参照箇所の [第X章 第X節 p.Y] を明記してください。

また、回答の末尾には必ず「---」で区切り、実際に回答作成で参照した資料を以下の【出力フォーマット】に従って章・節ごとにページ番号をグルーピングして一覧化してください。
回答の「---」より上の内容には、章・節・ページ番号は記載しないでください。(回答の見やすさ重視のため)

【出力フォーマットのルール】:
1. 同じ「章・節」の資料が複数ある場合は、1行にまとめてページ番号を「p.Y1, Y2」の形式で昇順（小さい順）に並べてください。
2. 回答作成に利用しなかった無関係な資料はリストに含めないでください。

---
**📚 参照した資料:**
- [章名] [節名] [p.88, 226, 269]
- [章名] [節名] [p.328]

【資料】:
{context}

【質問】:
{question}

【回答】:"""

# 3. チャットイベントハンドラ (v1で動作確認済みの安定仕様)
def chat_handler(message, history, search_mode):
    # 履歴（history）の初期化
    dict_history = list(history) if history else []

    if 'vector_store' not in globals() or 'bm25' not in globals():
        dict_history.append({"role": "user", "content": message})
        dict_history.append({"role": "assistant", "content": "⚠️ 先に検索エンジン構築セルを実行してください。"})
        return dict_history

    if not message.strip():
        return dict_history

    # 検索の実行（上位4件取得）
    if search_mode == "キーワード検索 (BM25)":
        kw_scores = bm25.get_scores(tokenize_jp(message))
        top_idx = sorted(range(len(kw_scores)), key=lambda i: kw_scores[i], reverse=True)[:4]
        hits = [{'doc': chunks[i], 'kw_rank': r, 'vec_rank': None} for r, i in enumerate(top_idx, 1)]
    elif search_mode == "ベクトル検索 (ChromaDB)":
        vec_res = vector_store.similarity_search_with_relevance_scores(message, k=4)
        hits = [{'doc': doc, 'kw_rank': None, 'vec_rank': r} for r, (doc, _) in enumerate(vec_res, 1)]
    else:
        hits = hybrid_search(message, top_k=4)

    # 資料テキストの結合
    context_parts = []
    for i, h in enumerate(hits, 1):
        m = h['doc'].metadata
        loc = f"{m.get('chapter', '')} {m.get('section', '')} [p.{m.get('page')}]"
        context_parts.append(f"--- [資料{i}: {loc}] ---\n{h['doc'].page_content}")

    # =================================================================
    # 【変更点】Python側での sources_line の強制結合をやめ、
    # Geminiがプロンプトの指示に従って出力した回答をそのまま採用する
    # =================================================================
    chain = PromptTemplate.from_template(prompt_tmpl) | llm | StrOutputParser()
    reply = chain.invoke({"context": "\n\n".join(context_parts), "question": message})

    # メッセージ履歴に追加して返却
    dict_history.append({"role": "user", "content": message})
    dict_history.append({"role": "assistant", "content": reply})
    return dict_history

# 4. Gradio UIの構築と起動 (Colab環境で型エラーの出ない安全な構成)
with gr.Blocks(title="2024年版ものづくり白書 RAG (v2)") as demo:
    gr.Markdown("# 📘 2024年版ものづくり白書 RAG チャットボット (v2)")
    gr.Markdown("**3段階検索切替: ① キーワード検索(BM25) / ② ベクトル検索(ChromaDB) / ③ ハイブリッド検索(RRF)**")

    mode_select = gr.Radio(
        choices=["ハイブリッド検索 (RRF統合) [推奨]", "キーワード検索 (BM25)", "ベクトル検索 (ChromaDB)"],
        value="ハイブリッド検索 (RRF統合) [推奨]",
        label="検索方式の選択"
    )
    chatbot_ui = gr.Chatbot(height=480, label="対話履歴 (出典付き回答)")
    msg = gr.Textbox(placeholder="白書について質問を入力 (例: 価格転嫁の進捗状況と課題について教えてください)...", label="質問入力")

    with gr.Row():
        send_btn = gr.Button("送信", variant="primary", scale=4)
        clear_btn = gr.Button("対話クリア", scale=1)

    # 送信時のイベント設定 (v1と同等)
    send_btn.click(chat_handler, [msg, chatbot_ui, mode_select], chatbot_ui)
    send_btn.click(lambda: "", None, msg)

    msg.submit(chat_handler, [msg, chatbot_ui, mode_select], chatbot_ui)
    msg.submit(lambda: "", None, msg)

    clear_btn.click(lambda: [], None, chatbot_ui)

demo.launch(share=True, debug=True)

## 5. 【コード単体検証】Geminiによる出典付き回答生成テスト
UIを介さず、Pythonコードから直接クエリを投入して検索結果やGeminiのプロンプト出力を詳細に確認・デバッグできます。
検索されたチャンク内容（Context）と、LLMによる出典番号の整合性を検証したい場合にご利用ください。

In [ ]:
# 質問内容を定義
test_question = "中小製造業における価格転嫁の現状と直面している課題について教えてください。"
print(f"❓ テスト質問: {test_question}\n")

# 1. ハイブリッド検索を実行
hits = hybrid_search(test_question, top_k=4)
print(f"🔍 ハイブリッド検索で取得した上位チャンク ({len(hits)}件):")
context_parts = []
for i, h in enumerate(hits, 1):
    m = h['doc'].metadata
    loc = f"{m.get('chapter', '')} {m.get('section', '')} (p.{m.get('page')})"
    print(f"  [{i}] {loc} - 抜粋: {h['doc'].page_content[:60]}...")
    context_parts.append(f"--- [資料{i}: {loc}] ---\n{h['doc'].page_content}")

# 2. プロンプト投入と回答生成
context_text = "\n\n".join(context_parts)
print("\n🤖 Gemini 3.8 Flash にプロンプトを投入中...")
answer = chain.invoke({"context": context_text, "question": test_question})

print("\n" + "=" * 60)
print("💡 回答結果 (出典明記):")
print("=" * 60)
print(answer)